# Trenowanie modeli — Książki (Book-Crossing)

## Cel tego notebooka
Trenowanie i porównanie modeli regresji liniowej i logistycznej na danych
z datasetu Book-Crossing. Pipeline jest identyczny jak dla filmów (03_model.ipynb) —
ten sam kod, te same metryki, inny typ treści.

Modele zostaną zapisane jako pliki .pkl do użycia w backendzie FastAPI.

📄 Szczegółowe wyjaśnienie: docs/15_books_model.md

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, precision_recall_curve

os.chdir(os.path.expanduser('~/magisterka'))
print('Working directory:', os.getcwd())

## Wczytanie przetworzonych danych

Wczytujemy dane przygotowane w notebooku 06_books_preprocessing.ipynb.
Dane są już przeskalowane (StandardScaler) i podzielone na zbiór treningowy i testowy.

In [ ]:
X_train = pd.read_csv('data/books_processed/X_train_books.csv')
X_test  = pd.read_csv('data/books_processed/X_test_books.csv')
y_train = pd.read_csv('data/books_processed/y_train_books.csv').squeeze()
y_test  = pd.read_csv('data/books_processed/y_test_books.csv').squeeze()

FEATURE_COLS = joblib.load('backend/model_books/feature_cols_books.pkl')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape},  y_test:  {y_test.shape}')
print(f'Cechy: {FEATURE_COLS}')

## Model bazowy (Baseline)

Baseline to najprostszy możliwy model — zawsze przewiduje średnią ocenę
ze zbioru treningowego. Służy jako punkt odniesienia:
każdy sensowny model musi być od niego lepszy.

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae  = mean_absolute_error(y_test, baseline_pred)

print(f'Baseline (średnia={y_train.mean():.4f})')
print(f'RMSE: {baseline_rmse:.4f}')
print(f'MAE:  {baseline_mae:.4f}')

## Regresja liniowa, Ridge i Lasso

Trenujemy trzy warianty regresji liniowej:
- **Linear Regression** — klasyczna regresja bez regularyzacji
- **Ridge** — regularyzacja L2 (karze za duże współczynniki)
- **Lasso** — regularyzacja L1 (może zerować współczynniki, selekcja cech)

Oceny w datasecie są w skali 1–10 (nie 1–5 jak filmy) —
przycinamy predykcje do zakresu 1–10.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, clip_min=1, clip_max=10):
    model.fit(X_tr, y_tr)
    pred = np.clip(model.predict(X_te), clip_min, clip_max)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae  = mean_absolute_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    print(f'{name}: RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}')
    return model, pred, rmse, mae, r2

lr,    pred_lr,    rmse_lr,    mae_lr,    r2_lr    = evaluate('Linear Regression', LinearRegression(), X_train, y_train, X_test, y_test)
ridge, pred_ridge, rmse_ridge, mae_ridge, r2_ridge = evaluate('Ridge',             Ridge(alpha=1.0),   X_train, y_train, X_test, y_test)
lasso, pred_lasso, rmse_lasso, mae_lasso, r2_lasso = evaluate('Lasso',             Lasso(alpha=0.01),  X_train, y_train, X_test, y_test)

## Tabela porównawcza modeli regresji

Zestawienie wszystkich modeli w jednej tabeli dla łatwego porównania.

In [ ]:
results = pd.DataFrame([
    {'Model': 'Baseline',           'RMSE': round(baseline_rmse, 4), 'MAE': round(baseline_mae, 4),  'R²': 0.0},
    {'Model': 'Linear Regression',  'RMSE': round(rmse_lr, 4),    'MAE': round(mae_lr, 4),    'R²': round(r2_lr, 4)},
    {'Model': 'Ridge',              'RMSE': round(rmse_ridge, 4), 'MAE': round(mae_ridge, 4), 'R²': round(r2_ridge, 4)},
    {'Model': 'Lasso',              'RMSE': round(rmse_lasso, 4), 'MAE': round(mae_lasso, 4), 'R²': round(r2_lasso, 4)},
])
display(results)

## Regresja logistyczna z threshold tuningiem

Przekształcamy problem na klasyfikację binarną:
ocena >= 7 (w skali 1–10) = "polubi" (1), ocena < 7 = "nie polubi" (0).
Próg 7 odpowiada ocenie 3.5/5 — analogicznie do progu 4/5 użytego dla filmów.

Threshold tuning: szukamy progu prawdopodobieństwa który maksymalizuje F1.

In [ ]:
y_train_bin = (y_train >= 7).astype(int)
y_test_bin  = (y_test  >= 7).astype(int)

print(f'Pozytywnych (>=7) w train: {y_train_bin.sum()} ({y_train_bin.mean():.1%})')
print(f'Pozytywnych (>=7) w test:  {y_test_bin.sum()}  ({y_test_bin.mean():.1%})')

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train_bin)

proba = log_reg.predict_proba(X_test)[:, 1]
auc   = roc_auc_score(y_test_bin, proba)
print(f'\nAUC-ROC: {auc:.4f}')

# threshold tuning
precisions, recalls, thresholds = precision_recall_curve(y_test_bin, proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx  = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx])

print(f'Optymalny próg: {optimal_threshold:.4f}')
print(f'F1 przy optymalnym progu: {f1_scores[best_idx]:.4f}')

## Zapis modeli i artefaktów

Zapisujemy wszystkie modele i artefakty do folderu `backend/model_books/`.
Backend FastAPI wczyta je przy starcie serwera.

In [ ]:
import json

joblib.dump(lr,      'backend/model_books/linear_model_books.pkl')
joblib.dump(ridge,   'backend/model_books/ridge_model_books.pkl')
joblib.dump(lasso,   'backend/model_books/lasso_model_books.pkl')
joblib.dump(log_reg, 'backend/model_books/logistic_model_books.pkl')

with open('backend/model_books/optimal_threshold_books.json', 'w') as f:
    json.dump({'optimal_threshold': optimal_threshold}, f)

print('Zapisano modele:')
print('  backend/model_books/linear_model_books.pkl')
print('  backend/model_books/ridge_model_books.pkl')
print('  backend/model_books/lasso_model_books.pkl')
print('  backend/model_books/logistic_model_books.pkl')
print('  backend/model_books/optimal_threshold_books.json')
print(f'\nOptymalny próg logistycznej: {optimal_threshold:.4f}')